<a href="https://colab.research.google.com/github/somendrew/LMS_Image_Classification/blob/main/LMS_Offical_Image_Stealth_Scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install -q torch torchvision transformers pillow

In [3]:
import pandas as pd

In [4]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Get credentials and authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

In [5]:
# # Open the sheet by its name
# spreadsheet = gc.open('  P0.1 Sports {Exploratory analysis} || FIFA || Somendra ')

# # Select the first worksheet (tab)
# worksheet = spreadsheet.sheet1

In [6]:
spreadsheet = gc.open('test_set_LMS')

worksheet = spreadsheet.worksheet('Copy of Sheet3')

In [7]:
worksheet

<Worksheet 'Copy of Sheet3' id:1337877554>

In [8]:
# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to a DataFrame (using the first row as headers)
df = pd.DataFrame(rows[1:], columns=rows[0])

# View the first few rows
df.head()

,MID,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Accuracy,Screenshot,Comment,Image Status,Assigned To,Status,Date of analysis
0,,hume,Tommaso Ravaglioli,,,,,,,,,,,,,,,,,
1,,hume,Adham El Idrissi,,,,,,,,,,,,,,,,,
2,,hume,Andrej Kotnik,,,,,,,,,,,,,,,,,
3,,hume,David,,,,,,,,,,,,,,,,,
4,,hume,Rahim Alhassane,,,,,,,,,,,,,,,,,


In [9]:
#Taking sample df

new_df = df[:]
new_df.shape

(60, 20)

In [10]:
new_df.head()

,MID,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Accuracy,Screenshot,Comment,Image Status,Assigned To,Status,Date of analysis
0,,hume,Tommaso Ravaglioli,,,,,,,,,,,,,,,,,
1,,hume,Adham El Idrissi,,,,,,,,,,,,,,,,,
2,,hume,Andrej Kotnik,,,,,,,,,,,,,,,,,
3,,hume,David,,,,,,,,,,,,,,,,,
4,,hume,Rahim Alhassane,,,,,,,,,,,,,,,,,


## Image Processor

In [11]:
import torch
from PIL import Image
from transformers import (
    DetrImageProcessor, DetrForObjectDetection,
    BlipProcessor, BlipForQuestionAnswering
)
from io import BytesIO # Added this import

# ─── Load models once ──────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"

# Person counter — DETR
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device)
detr_model.eval()

# Visual Q&A — BLIP
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)
blip_model.eval()

# ─── Helper functions ──────────────────────────────────────────

def count_people(image: Image.Image, threshold: float = 0.85) -> int:
    """Returns number of people detected in the image."""
    inputs = detr_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = detr_model(**inputs)

    results = detr_processor.post_process_object_detection(
        outputs,
        target_sizes=torch.tensor([image.size[::-1]]),
        threshold=threshold
    )[0]

    labels = results["labels"].tolist()
    # COCO label 1 = person
    return labels.count(1)


def ask_blip(image: Image.Image, question: str) -> bool:
    """Ask a yes/no question about the image using BLIP VQA."""
    inputs = blip_processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        output = blip_model.generate(**inputs)
    answer = blip_processor.decode(output[0], skip_special_tokens=True).strip().lower()
    return answer.startswith("yes")


# ─── Main analyzer ──────────────────────────────────────────
def analyze_profile_photo(image_source: str) -> dict:
    if image_source.startswith("http://") or image_source.startswith("https://"):
        response = requests.get(image_source)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_source).convert("RGB")

    person_count = count_people(image)
    exactly_one_person = (person_count == 1)

    front_facing = ask_blip(
        image,
        "Is the person's face directly facing the camera?"
    )
    consistent_background = ask_blip(
        image,
        "Is the background plain, uniform ?"
    )
    official_style = ask_blip(
        image,
        "Does this look like a professional ID photo with good lighting and sharp focus?"
    )

    passes = all([exactly_one_person, front_facing, consistent_background, official_style])

    return {
        "exactly_one_person": exactly_one_person,
        "person_count": person_count,
        "front_facing": front_facing,
        "consistent_background": consistent_background,
        "official_style": official_style,
        "passes": passes,
    }

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Scraper

## Iterate through DataFrame and fill the links

### Cleaning the entity name


In [12]:

import unicodedata

def simplify_name(text):
    # Normalize the unicode to decomposition form (NFD)
    # This separates "ā" into "a" + "◌̄"
    normalized = unicodedata.normalize('NFD', text)

    # Filter out the non-spacing mark characters (the accents)
    simplified = "".join(c for c in normalized if unicodedata.category(c) != 'Mn')

    return simplified

In [13]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import time
player_images = []

def stealth_scrape(entity):
    url = f"https://www.gettyimages.com/photos/{entity.replace(' ', '-')}"

    response = requests.get(url, impersonate="chrome120")

    images_found = []  # Local list instead of global

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        images = soup.find_all('img')
        for img in images[:20]:
            src = img.get('src')
            if src:  # Also guard against None src values
                images_found.append(src)
    else:
        print(f"Blocked! Status Code: {response.status_code}")

    return images_found  # Always return a list (empty if blocked)


# Iterate through the DataFrame
print("Starting bulk scrape...")

for index, row in new_df.iterrows():

    uncleaned_entity = row['Athlete Name']
    entity = simplify_name(uncleaned_entity)

    query = f'{entity} poses for a Portrait'

    print(f"Processing: {entity}")

    # Fetch links
    player_images = stealth_scrape(query)

    # Splitting entity to check for individual keywords (case-insensitive) present in the link
    keywords = [k.lower() for k in entity.split()]

    #Defining number of getty columns we have in sheet
    max_columns = 5

    filtered_images = [ link for link in player_images if all(word in link.lower() for word in keywords)]
    print(f"Total images found: {len(filtered_images)}")

    vllm_images = [img for img in filtered_images if analyze_profile_photo(img)["passes"]] #true if passes all test cases
    print(f"Total images after vllm_processing: {len(vllm_images)}")

    # Images we need (5)
    images_to_save = vllm_images[:max_columns]

    # Fill the dataframe columns
    for i, link in enumerate(images_to_save):
        new_df.at[index, f'Getty Image link {i+1}'] = link

    # Respectful delay to avoid IP bans
    time.sleep(2)



Starting bulk scrape...
Processing: Tommaso Ravaglioli
Total images found: 2
Total images after vllm_processing: 2
Processing: Adham El Idrissi
Total images found: 0
Total images after vllm_processing: 0
Processing: Andrej Kotnik
Total images found: 0
Total images after vllm_processing: 0
Processing: David
Total images found: 5
Total images after vllm_processing: 3
Processing: Rahim Alhassane
Total images found: 0
Total images after vllm_processing: 0
Processing: Thiago Borbas
Total images found: 0
Total images after vllm_processing: 0
Processing: Camilo Puentes
Total images found: 0
Total images after vllm_processing: 0
Processing: Charlie Tasker
Total images found: 0
Total images after vllm_processing: 0
Processing: Etienne Eto'o
Total images found: 0
Total images after vllm_processing: 0
Processing: Kristian Bilovar
Total images found: 0
Total images after vllm_processing: 0
Processing: Altin Kryeziu
Total images found: 0
Total images after vllm_processing: 0
Processing: Sergio Hern

In [14]:
new_df

,MID,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Accuracy,Screenshot,Comment,Image Status,Assigned To,Status,Date of analysis
0,,hume,Tommaso Ravaglioli,https://media.gettyimages.com/id/1255214247/ph...,,https://media.gettyimages.com/id/1255214175/ph...,,,,,,,,,,,,,,
1,,hume,Adham El Idrissi,,,,,,,,,,,,,,,,,
2,,hume,Andrej Kotnik,,,,,,,,,,,,,,,,,
3,,hume,David,https://media.gettyimages.com/id/74254269/phot...,,https://media.gettyimages.com/id/2220530404/ph...,,https://media.gettyimages.com/id/2157322287/ph...,,,,,,,,,,,,
4,,hume,Rahim Alhassane,,,,,,,,,,,,,,,,,
5,,hume,Thiago Borbas,,,,,,,,,,,,,,,,,
6,,hume,Camilo Puentes,,,,,,,,,,,,,,,,,
7,,hume,Charlie Tasker,,,,,,,,,,,,,,,,,
8,,hume,Etienne Eto'o,,,,,,,,,,,,,,,,,
9,,hume,Kristian Bilovar,,,,,,,,,,,,,,,,,


## Direct Update

In [15]:
# 1. Convert DataFrame back to a list of lists (including headers)
# .columns.tolist() gets the headers
# .values.tolist() gets the data rows
data_to_upload = [new_df.columns.tolist()] + new_df.values.tolist()

# 2. Update the sheet starting from cell A1
worksheet.update('A1', data_to_upload)

{'spreadsheetId': '1ZxpzmUjGdR_iyWA_qpIjNCgj-M9oaPn5v72AaCpJY6Q',
 'updatedRange': "'Copy of Sheet3'!A1:T61",
 'updatedRows': 61,
 'updatedColumns': 20,
 'updatedCells': 1220}

## Download

In [16]:
# from google.colab import files

# #  Save the file first
# new_df.to_csv('test_set2.csv', index=False)

# # Trigger the browser download
# files.download('test_set2.csv')